In [1]:
!pip install dagshub mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import dagshub
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("XGBoost")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())
print("✅ XGBoost version:", xgb.__version__)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=11a828e0-d25e-46fa-a19c-d258b27b8fd7&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=49898933993b8328614eb12f49fa3e1a8a00cbea69c66bedb0308664c3ef07a2




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

2026/05/01 20:33:12 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost' does not exist. Creating a new experiment.


✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow
✅ XGBoost version: 3.2.0


# **Cleaning**

In [3]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD   = 0.80
    ID_MISSING_THRESHOLD    = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    test_id.columns  = [c.replace('-', '_') for c in test_id.columns]
    train_id.columns = [c.replace('-', '_') for c in train_id.columns]

    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id,  on="TransactionID", how="left")

    y_train_xgb = train["isFraud"].copy()
    X_train_xgb = train.drop(columns=["isFraud", "TransactionID"])
    X_test_xgb  = test.drop(columns=["TransactionID"])

    del train, test, train_txn, train_id, test_txn, test_id
    gc.collect()

    id_like_cols  = [c for c in X_train_xgb.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train_xgb.columns if c not in id_like_cols]

    missing_ratio = X_train_xgb.isnull().mean()
    drop_txn      = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id       = [c for c in id_like_cols  if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing  = drop_txn + drop_id

    X_train_xgb = X_train_xgb.drop(columns=drop_missing)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in drop_missing if c in X_test_xgb.columns])

    near_constant_cols = []
    for col in X_train_xgb.columns:
        top_freq = X_train_xgb[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train_xgb = X_train_xgb.drop(columns=near_constant_cols)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in near_constant_cols if c in X_test_xgb.columns])

    for col in X_train_xgb.columns:
        if col not in X_test_xgb.columns:
            X_test_xgb[col] = np.nan
    X_test_xgb = X_test_xgb[X_train_xgb.columns]

    mlflow.log_param("txn_missing_threshold",   TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold",    ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)
    mlflow.log_metric("train_rows",             int(X_train_xgb.shape[0]))
    mlflow.log_metric("test_rows",              int(X_test_xgb.shape[0]))
    mlflow.log_metric("features_after_cleaning",int(X_train_xgb.shape[1]))
    mlflow.log_metric("dropped_high_missing",   len(drop_missing))
    mlflow.log_metric("dropped_near_constant",  len(near_constant_cols))
    mlflow.log_metric("fraud_rate",             float(y_train_xgb.mean()))

    print(f"X_train: {X_train_xgb.shape}")
    print(f"X_test:  {X_test_xgb.shape}")
    print(f"Fraud rate: {y_train_xgb.mean():.4f}")

    X_train_clean_xgb = X_train_xgb
    X_test_clean_xgb  = X_test_xgb
    y_train_clean_xgb = y_train_xgb

X_train: (590540, 353)
X_test:  (506691, 353)
Fraud rate: 0.0350
🏃 View run XGBoost_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/1dc117b6df284dc093b34b7515e1ebd0
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Engineering

In [4]:
with mlflow.start_run(run_name="XGBoost_FeatureEngineering"):
    X_train = X_train_clean_xgb.copy()
    X_test  = X_test_clean_xgb.copy()
    y_train = y_train_clean_xgb.copy()

    # log transform — TransactionAmt is heavily right-skewed
    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"]  = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    # cyclic hour encoding — fraud has strong time-of-day pattern
    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"]  = np.sin(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)
    X_test["hour_cos"]  = np.cos(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test  = X_test.drop(columns=["TransactionDT"],  errors="ignore")

    # ordinal encode categoricals
    # XGBoost handles ordinal-encoded cats fine — no one-hot needed
    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_param("cat_encoding",     "ordinal_train_mapping")
    mlflow.log_param("amt_log_transform", True)
    mlflow.log_param("cyclic_time_encoding", True)
    mlflow.log_metric("features_after_fe",   int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded",    len(cat_cols))

    print(f"X_train_fe: {X_train.shape}")
    print(f"X_test_fe:  {X_test.shape}")

    X_train_fe_xgb = X_train
    X_test_fe_xgb  = X_test
    y_train_fe_xgb = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run XGBoost_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/0adcf23ab760420283c887c6256cf7c1
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Selection

In [5]:
with mlflow.start_run(run_name="XGBoost_FeatureSelection"):
    X_train = X_train_fe_xgb.copy()
    X_test  = X_test_fe_xgb.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan).fillna(0)

    # drop constant columns
    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test  = X_test.drop(columns=const_cols,  errors="ignore")

    # drop highly correlated features — sampled for speed
    CORR_THRESHOLD = 0.98
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    sample_n = min(120_000, len(X_train))
    idx  = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test  = X_test.drop(columns=drop_corr,  errors="ignore")
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("corr_threshold",     CORR_THRESHOLD)
    mlflow.log_metric("dropped_const",     len(const_cols))
    mlflow.log_metric("dropped_high_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print(f"X_train_fs: {X_train.shape}")
    print(f"X_test_fs:  {X_test.shape}")

    X_train_final_xgb = X_train
    X_test_final_xgb  = X_test

X_train_fs: (590540, 301)
X_test_fs:  (506691, 301)
🏃 View run XGBoost_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/84d9029fd2424e018551927cd9ee1a2f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Training

In [6]:
# holdout split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final_xgb, y_train_fe_xgb,
    test_size=0.2, random_state=42, stratify=y_train_fe_xgb
)

# scale_pos_weight handles class imbalance in XGBoost
# it's the ratio of negative to positive samples
neg = int((y_tr == 0).sum())
pos = int((y_tr == 1).sum())
spw = round(neg / pos, 2)
print(f"Train size: {X_tr.shape} | Val size: {X_val.shape}")
print(f"scale_pos_weight = {spw}  (neg/pos = {neg}/{pos})")

Train size: (472432, 301) | Val size: (118108, 301)
scale_pos_weight = 27.58  (neg/pos = 455902/16530)


In [7]:
with mlflow.start_run(run_name="XGB_Baseline"):
    mlflow.log_param("n_estimators",    100)
    mlflow.log_param("max_depth",       6)
    mlflow.log_param("learning_rate",   0.3)
    mlflow.log_param("scale_pos_weight",1)
    mlflow.log_param("tree_method",     "hist")
    mlflow.log_param("device",          "cuda")
    mlflow.log_param("note",            "default params, no class balancing")

    clf = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        scale_pos_weight=1,
        tree_method="hist",
        device="cuda",
        eval_metric="auc",
        random_state=42
    )
    clf.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(train_auc - val_auc, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")
    print("  -> default lr=0.3, no class balancing. expecting decent baseline but room to improve.")

[Baseline] Train: 0.9588 | Val: 0.9391 | Gap: 0.0197
  -> default lr=0.3, no class balancing. expecting decent baseline but room to improve.
🏃 View run XGB_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/82ab57bbbf9b4688a051a2b0a595d9ae
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [8]:
# more trees without regularization → overfitting
# early stopping finds the sweet spot automatically
for n_est in [100, 300, 500]:
    with mlflow.start_run(run_name=f"XGB_nestimators_{n_est}"):
        mlflow.log_param("n_estimators",    n_est)
        mlflow.log_param("max_depth",       6)
        mlflow.log_param("learning_rate",   0.1)
        mlflow.log_param("scale_pos_weight",spw)
        mlflow.log_param("tree_method",     "hist")
        mlflow.log_param("device",          "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=n_est,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=spw,
            tree_method="hist",
            device="cuda",
            eval_metric="auc",
            random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        print(f"  n_estimators={n_est:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

  n_estimators=100  | Train: 0.9372 | Val: 0.9226 | Gap: 0.0146
🏃 View run XGB_nestimators_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/457924bcb5f4439e9d80f76d767ceef6
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_estimators=300  | Train: 0.9683 | Val: 0.9452 | Gap: 0.0230
🏃 View run XGB_nestimators_300 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/dd2b6a25503347f68b59468b2b4727b3
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_estimators=500  | Train: 0.9810 | Val: 0.9539 | Gap: 0.0271
🏃 View run XGB_nestimators_500 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/881f166b2284460bb0c620aaeb40d94d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [9]:
# shallow trees = underfitting, deep trees = overfitting
# key hyperparameter for XGBoost
depth_results = []
for depth in [3, 4, 6, 8, 10]:
    with mlflow.start_run(run_name=f"XGB_depth_{depth}"):
        mlflow.log_param("n_estimators",    300)
        mlflow.log_param("max_depth",       depth)
        mlflow.log_param("learning_rate",   0.1)
        mlflow.log_param("scale_pos_weight",spw)
        mlflow.log_param("tree_method",     "hist")
        mlflow.log_param("device",          "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=depth,
            learning_rate=0.1,
            scale_pos_weight=spw,
            tree_method="hist",
            device="cuda",
            eval_metric="auc",
            random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        depth_results.append({"max_depth": depth, "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"  depth={depth} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

depth_df = pd.DataFrame(depth_results).set_index("max_depth")
best_depth = int(depth_df["val_auc"].idxmax())
print(f"\n-> Best depth: {best_depth}")
print(depth_df.to_string())

  depth=3 | Train: 0.9162 | Val: 0.9076 | Gap: 0.0086
🏃 View run XGB_depth_3 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/6caa78acd3ba4a1590b5b2f90b76b42b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=4 | Train: 0.9347 | Val: 0.9211 | Gap: 0.0137
🏃 View run XGB_depth_4 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/fcfc46d80d3f4e52945624092e571a79
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=6 | Train: 0.9683 | Val: 0.9452 | Gap: 0.0230
🏃 View run XGB_depth_6 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/6991ac6b0f464e75ae1493c7fc970d29
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=8 | Train: 0.9887 | Val: 0.9584 | Gap: 0.0303
🏃 View run XGB_depth_8 at: https://dagshub.com/sophyris

In [10]:
# subsampling rows and columns per tree reduces overfitting
# similar idea to Random Forest but within boosting
for subsample, colsample in [(1.0, 1.0), (0.8, 0.8), (0.7, 0.7)]:
    label = f"sub{subsample}_col{colsample}"
    with mlflow.start_run(run_name=f"XGB_sampling_{label}"):
        mlflow.log_param("n_estimators",    300)
        mlflow.log_param("max_depth",       best_depth)
        mlflow.log_param("learning_rate",   0.1)
        mlflow.log_param("subsample",       subsample)
        mlflow.log_param("colsample_bytree",colsample)
        mlflow.log_param("scale_pos_weight",spw)
        mlflow.log_param("tree_method",     "hist")
        mlflow.log_param("device",          "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=best_depth,
            learning_rate=0.1,
            subsample=subsample,
            colsample_bytree=colsample,
            scale_pos_weight=spw,
            tree_method="hist",
            device="cuda",
            eval_metric="auc",
            random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        print(f"  {label} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

# best subsampling config — update manually based on output
best_subsample   = 0.8
best_colsample   = 0.8

  sub1.0_col1.0 | Train: 0.9976 | Val: 0.9638 | Gap: 0.0338
🏃 View run XGB_sampling_sub1.0_col1.0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/afc56784029449aa87568f3a55c3f74b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.8_col0.8 | Train: 0.9996 | Val: 0.9657 | Gap: 0.0339
🏃 View run XGB_sampling_sub0.8_col0.8 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/0bbdbce4e80148a59ecb2eab74bd6637
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.7_col0.7 | Train: 0.9996 | Val: 0.9642 | Gap: 0.0353
🏃 View run XGB_sampling_sub0.7_col0.7 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/169ec1ce362a425d97770f996439df23
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [11]:
# CV on a sample — full 590k would be too slow even on GPU for 5 folds
sample_idx = np.random.RandomState(42).choice(len(X_train_final_xgb), size=150_000, replace=False)
X_cv = X_train_final_xgb.iloc[sample_idx]
y_cv = y_train_fe_xgb.iloc[sample_idx]

with mlflow.start_run(run_name="XGB_CrossValidation_5fold"):
    mlflow.log_param("n_estimators",    300)
    mlflow.log_param("max_depth",       best_depth)
    mlflow.log_param("learning_rate",   0.1)
    mlflow.log_param("subsample",       best_subsample)
    mlflow.log_param("colsample_bytree",best_colsample)
    mlflow.log_param("scale_pos_weight",spw)
    mlflow.log_param("cv_folds",        5)
    mlflow.log_param("cv_sample_size",  150_000)

    clf_cv = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=best_depth,
        learning_rate=0.1,
        subsample=best_subsample,
        colsample_bytree=best_colsample,
        scale_pos_weight=spw,
        tree_method="hist",
        device="cuda",
        eval_metric="auc",
        random_state=42
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf_cv, X_cv, y_cv, cv=cv, scoring="roc_auc")

    for i, score in enumerate(cv_scores):
        mlflow.log_metric("fold_auc", round(score, 5), step=i)

    mlflow.log_metric("cv_mean_auc", round(cv_scores.mean(), 5))
    mlflow.log_metric("cv_std_auc",  round(cv_scores.std(),  5))

    print(f"CV folds: {[round(s, 4) for s in cv_scores]}")
    print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")

CV folds: [np.float64(0.9345), np.float64(0.942), np.float64(0.9337), np.float64(0.9371), np.float64(0.9288)]
Mean: 0.9352 | Std: 0.0043
🏃 View run XGB_CrossValidation_5fold at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/4a36dc1296ac41c587d77d640d398025
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [12]:
with mlflow.start_run(run_name="XGB_Final_Pipeline") as run:
    mlflow.log_param("n_estimators",    500)
    mlflow.log_param("max_depth",       best_depth)
    mlflow.log_param("learning_rate",   0.05)
    mlflow.log_param("subsample",       best_subsample)
    mlflow.log_param("colsample_bytree",best_colsample)
    mlflow.log_param("scale_pos_weight",spw)
    mlflow.log_param("tree_method",     "hist")
    mlflow.log_param("device",          "cuda")
    mlflow.log_param("trained_on",      "full_train_set")
    mlflow.log_param("note",            "lower lr=0.05, more trees for final model")

    final_clf = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=best_depth,
        learning_rate=0.05,
        subsample=best_subsample,
        colsample_bytree=best_colsample,
        scale_pos_weight=spw,
        tree_method="hist",
        device="cuda",
        eval_metric="auc",
        random_state=42
    )
    final_clf.fit(X_train_final_xgb, y_train_fe_xgb)

    val_auc = roc_auc_score(y_val, final_clf.predict_proba(X_val)[:, 1])
    mlflow.log_metric("val_auc", round(val_auc, 5))

    # wrap in Pipeline so inference notebook can load and call predict directly
    final_pipe = Pipeline([("clf", final_clf)])

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="xgboost_pipeline",
        registered_model_name="XGBoost_FraudDetection"
    )

    print(f"Final Pipeline Val AUC: {val_auc:.4f}")
    print(f"Run ID: {run.info.run_id}")

2026/05/01 20:55:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 20:55:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_FraudDetection'.
2026/05/01 20:55:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection, version 1
Created version '1' of model 'XGBoost_FraudDetection'.


Final Pipeline Val AUC: 0.9981
Run ID: 87ebd8977e2c4ed7917b8140b8c091cf
🏃 View run XGB_Final_Pipeline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/87ebd8977e2c4ed7917b8140b8c091cf
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
